# CRBAについて

ロボットの運動方程式を以下とする。

$$
M(\boldsymbol{q}) \ddot{\boldsymbol{q}} + h (\boldsymbol{q}, \dot{\boldsymbol{q}}) = \boldsymbol{\tau}
$$

- $M(\boldsymbol{q})$ : 質量行列 ($n \times n$)
- $h (\boldsymbol{q}, \dot{\boldsymbol{q}})$ : コリオリ・遠心力、重力などのバイアストルク
- $\boldsymbol{\tau}$ : 関節トルク

CRBAは質量行列 $M(\boldsymbol{q})$ を効率的に計算するアルゴリズムである。$h(\boldsymbol{q}, \dot{\boldsymbol{q}})$は計算しない。

### RNEA, ABA , CRBAの違い

|アルゴリズム|入力|出力|
|:---|:---|:---|
|RENA| $\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}$ | $\boldsymbol{\tau}$ |
|ABA| $\boldsymbol{q}, \dot{\boldsymbol{q}}, \boldsymbol{\tau}$ | $\ddot{\boldsymbol{q}}$ |
|CRBA| $\boldsymbol{q}$ | $M(\boldsymbol{q})$ |



## 1. 単位ベクトル法

RNEAを使っても質量行列を得ることはできる。

例えば、運動方程式上で以下のように$h$を消去する操作を行う。

$$
\dot{\boldsymbol{q}} = \boldsymbol{g} = 0 \quad \rightarrow \quad h(\boldsymbol{q}, \dot{\boldsymbol{q}}) 
$$

すると、以下の式となる。

$$
\tau = M(\boldsymbol{q})\ddot{\boldsymbol{q}}
$$

この式を展開すると例えば以下のようなものである。

$$
\begin{bmatrix}
\tau_1 \\ \tau_2
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
\ddot{q}_1 \\ \ddot{q_2}
\end{bmatrix}
$$

各関節を順に単位加速度で動かすことを考える。$e_j$は第$j$成分だけが1の単位ベクトルである。

$$
\ddot{\boldsymbol{q}} = e_j
$$

すると、上記の例に当てはめると、

- $e_1$

$$
\begin{bmatrix}
m_{11}(q_1,q_2) \\ m_{21}(q_1,q_2)
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
1 \\ 0
\end{bmatrix}
$$

- $e_2$

$$
\begin{bmatrix}
m_{12}(q_1,q_2) \\ m_{22}(q_1,q_2)
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
0 \\ 1
\end{bmatrix}
$$

RNEAは$\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}$ から$\boldsymbol{\tau}$を効率的に求める方法である。重力を含めると以下の関係である。

$$
\boldsymbol{\tau} = \text{RNEA}(\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}, \boldsymbol{g})
$$

よって、RNEAを以下のように以下のように単位加速度ベクトル$e_j$で計算することを$n$回行えば質量行列全体を求めることが出来る。

$$
M_{:j} = \text{RNEA}(\boldsymbol{q}, 0, e_j, \boldsymbol{g}=0)
$$

一方、CRBAはComposite Rigid Body Inertiaを利用して、より効率的に質量行列を計算する。

## 2. Composite Rigid Body Inertia について

例えば次のような2 link のアームを考える。$q_1,q_2$は回転の正方向を示している。

<p align="center">
<img src="./images/2link_images.png" style="width:50%;" />
</p>

各リンクの[Spatial Inertia](../../ABA/Basic/spatial_vector.ipynb) を以下とする。

$$
{^1}\mathcal{I}_1, \quad {^2}\mathcal{I}_2
$$

$J_1$を動かすと、Link1 だけでなく Link2 も運動する。したがって、$J_1$から見た慣性にはLink2の慣性も含まれる。そこで、Link2の Spatial Inertia をLink1の座標系へ変換して合成する。

$$
\mathcal_{I}^C = {^1}\mathcal{I}_1 + \left({^1}X_{2}\right)^T \mathcal{I}_2 {^1}X_{2}
$$

- $\mathcal{I}^C$ : Link1をベースとする部分機構のComposite Rigid Body Inertia
- ${^1}X_{2}$ : Link2からLink1へ座標系の変換し、基準点をLink2原点からLink1原点へ変更する Spatial Motion Transform

この式は、Link2をJ2で固定し($\dot{q}=\ddot{q}=0$)、Link1と一体の剛体として扱ったときのSpatial Inertia を表している。

## 3. ABAで用いたABIとの違い

ABAでは次のような子リンクの関節加速度を消去し、関節自由度の影響を反映した等価慣性は以下となることを示した。

$$
\bar{\boldsymbol{I}}_i^A = \boldsymbol{I}_i^A - \boldsymbol{U}_i d_i^{-1} {\boldsymbol{U}_i}^T
$$

In [ ]:
##